In [0]:
from pyspark.sql.functions import col, upper, initcap, trim, when, to_timestamp

spark.sql("CREATE SCHEMA IF NOT EXISTS nyc311_project.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS nyc311_project.quarantine")

bronze = spark.table("nyc311_project.bronze.service_requests")

drop_cols = [
    "bridge_highway_name", "bridge_highway_direction", "road_ramp",
    "bridge_highway_segment", "vehicle_type", "taxi_pick_up_location",
    "taxi_company_borough", "due_date", "facility_type"
]
cleaned = bronze.drop(*drop_cols)

# Clean "nan"/"None"/"" placeholders across ALL string columns first
string_cols = [f.name for f in cleaned.schema.fields if str(f.dataType) == "StringType()"]
for c in string_cols:
    cleaned = cleaned.withColumn(c, when(col(c).isin("nan", "None", ""), None).otherwise(col(c)))

# Standardize casing
cleaned = (cleaned
    .withColumn("complaint_type", initcap(trim(col("complaint_type"))))
    .withColumn("descriptor", initcap(trim(col("descriptor"))))
    .withColumn("borough", upper(trim(col("borough"))))
    .withColumn("status", initcap(trim(col("status"))))
)

# "Unspecified" -> real null
cleaned = (cleaned
    .withColumn("borough", when(col("borough") == "UNSPECIFIED", None).otherwise(col("borough")))
    .withColumn("status", when(col("status") == "Unspecified", None).otherwise(col("status")))
)

# Timestamps
cleaned = (cleaned
    .withColumn("created_ts", to_timestamp("created_date"))
    .withColumn("closed_ts", to_timestamp("closed_date"))
    .drop("created_date", "closed_date")
)

# Quarantine split
is_unlocatable = col("incident_zip").isNull() & col("latitude").isNull() & col("city").isNull()
quarantine_df = cleaned.filter(is_unlocatable)
good_df = cleaned.filter(~is_unlocatable)

print(f"Total: {cleaned.count()}, Good: {good_df.count()}, Quarantined: {quarantine_df.count()}")

Total: 200000, Good: 199053, Quarantined: 947


In [0]:
good_df.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc311_project.silver.service_requests")

quarantine_df.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc311_project.quarantine.service_requests_unlocatable")

print("Silver rows:", spark.table("nyc311_project.silver.service_requests").count())
print("Quarantine rows:", spark.table("nyc311_project.quarantine.service_requests_unlocatable").count())

Silver rows: 199053
Quarantine rows: 947


In [0]:
display(spark.table("nyc311_project.quarantine.service_requests_unlocatable")
        .select("unique_key", "complaint_type", "incident_zip", "city", "latitude", "borough")
        .limit(20))

unique_key,complaint_type,incident_zip,city,latitude,borough
61655236,Highway Condition,null,null,null,STATEN ISLAND
61657640,Ferry Inquiry,null,null,null,null
61658887,Street Condition,null,null,null,null
61656821,Street Condition,null,null,null,null
61654166,Street Condition,null,null,null,null
61656580,Highway Condition,null,null,null,BROOKLYN
61660882,Traffic Signal Condition,null,null,null,null
61659652,Ferry Inquiry,null,null,null,null
61657639,Ferry Inquiry,null,null,null,null
61655490,Highway Condition,null,null,null,QUEENS


In [0]:
display(spark.table("nyc311_project.silver.service_requests").select("complaint_type").distinct().limit(30))

complaint_type
Blocked Driveway
Noise - Residential
Noise - Commercial
Illegal Parking
Noise - Street/sidewalk
Disorderly Youth
Illegal Fireworks
Street Light Condition
Food Establishment
Unsanitary Condition
